# README
- Author: RUEI-HSIANG, CHANG (B12303030)，Codex協助
- Created At: 2026-10-19
- Last Modified At: 2026-10-19
- Purpose: 從已保存的官方人物頁擷取明確學經歷，排除只有title的頁框；補陳素月2015官方公報生日，保留來源年代及申報層級。
- Input: 固定hash的pre_government_round4快照三表、縣市長round3、學歷來源線索及兩份來源manifest；逐檔路徑與SHA見PINS。
- Output: government_profiles/evidence/access_review_round4；縣市長round4；更新current主表、逐欄表與議員背景表，均為資料蒐集盤點而非最終分析樣本。
- Known limitations: 7位新人物頁的原文不等於學位認證；杜文卿歷史性別與曾參選證據另存，不混入690位2022身分支持欄。戴光宗、曾亮哲僅頁框，不將第三方學歷抄為官方原文。未知與存取失敗不填零。Meta與貼文蒐集暫緩。


版本註記：本輪原current檔已封存至snapshots/pre_bulletin_round5_20261019，最新current改由council_bulletin_round5_20261019.ipynb產出。

In [1]:
from pathlib import Path
import hashlib,json,re,unicodedata
import pandas as pd
from bs4 import BeautifulSoup
import pypdfium2 as pdfium
ROOT=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'AGENTS.md').exists() and (p/'templates').is_dir())
def sha(p):return hashlib.sha256(p.read_bytes()).hexdigest()
def norm(s):return re.sub(r'\s+','',unicodedata.normalize('NFKC',s)).replace('台','臺')
def read(p):return pd.read_csv(ROOT/p,dtype=str).fillna('')
PINS={'data/processed/snapshots/pre_government_round4_20261019/candidate_master_current.csv': 'bc3a9a7ef0aad7b2ac792581295b653e116dc630d20164fb4def968bf51d52a5', 'data/processed/snapshots/pre_government_round4_20261019/candidate_fields_current.csv': 'df2314986ed5eed973dc35d6d7c6f556d99ea55a2809f86fd5f1045313ec6894', 'data/processed/snapshots/pre_government_round4_20261019/council_background_20261019.csv': 'e64466d2e9f59bf9e2344fc7491b7ae585c00d26351b3b8d161cc468a5757430', 'data/processed/candidates_2026_enriched_round3_20261019.csv': 'e28951160f82504eb5b5d61777995ceb78a9f94bae89a724e4a9238e4c08192d', 'data/processed/snapshots/source_extraction_20261019/council_education_source_leads_20261019.csv': 'adad3affa587d85301205160f25b83770a5abb7cc8c9d8f84a4a9683c78da349', 'data/raw/background_expansion_20261019/capture_manifest.json': '12479a499c4052a55eb5f84758a661a294d60f436e45bfcf6329ac2a270f6aab', 'data/raw/government_profiles_round4_20261019/manifest.json': 'cd41b4cb7473fcc0b65497b85e8f460c9ed31b49990f298246b27ae308377a72'}
for p,h in PINS.items():assert sha(ROOT/p)==h,p
base='data/processed/snapshots/pre_government_round4_20261019/'
master=read(base+'candidate_master_current.csv');fields=read(base+'candidate_fields_current.csv');council=read(base+'council_background_20261019.csv')
mayor=read('data/processed/candidates_2026_enriched_round3_20261019.csv')
old_m=master.copy();old_f=fields.copy();old_c=council.copy();old_y=mayor.copy()
leads=read('data/processed/snapshots/source_extraction_20261019/council_education_source_leads_20261019.csv')
old_manifest=json.loads((ROOT/'data/raw/background_expansion_20261019/capture_manifest.json').read_text());new_manifest=json.loads((ROOT/'data/raw/government_profiles_round4_20261019/manifest.json').read_text())
lookup={r['url']:r for r in old_manifest+new_manifest}
for r in new_manifest:assert r['status']=='200' and sha(ROOT/r['path'])==r['sha256']
def soup_for(r):
 assert sha(ROOT/r['path'])==r['sha256']
 s=BeautifulSoup((ROOT/r['path']).read_bytes().decode('utf-8'),'html.parser')
 for node in s(['script','style','noscript']):node.decompose()
 return s
profiles=[];evidence=[];audit=[]
def evidence_row(p,field,value,url,locator,year,tier,identity='source_name_county_context_reviewed_independent_linkage_pending'):
 r=lookup[url]
 assert sha(ROOT/r['path'])==r['sha256']
 evidence.append(dict(candidate_key=p['candidate_key'],candidate_name=p['candidate_name'],county_city=p['county_city'],field=field,value_raw=value,source_url=url,source_file=r['path'],source_sha256=r['sha256'],source_locator=locator,source_year=year,evidence_tier=tier,identity_status=identity,checked_date='2026-10-19'))
def update(key,field,value,url,status,note):
 mask=fields.candidate_key.eq(key)&fields.field.eq(field);assert mask.sum()==1
 fields.loc[mask,['existing_value','evidence','status','note','next_source']]=[value,url,status,note,'原始來源與候選人身分／來源年代交叉核對；非學位認證']
 mask=master.candidate_key.eq(key);assert mask.sum()==1
 master.loc[mask,field+'_status']=status
names=['廖詒綉','戴德滿','何秀英','陳靖雅','謝昌年','杜文卿','詹錦章','戴光宗','曾亮哲']
for name in names:
 source=leads[leads.candidate_name.eq(name)].iloc[0].to_dict();r=lookup[source['fetch_url']];s=soup_for(r);text=s.get_text(' ',strip=True)
 p=council[council.candidate_key.eq(source['candidate_key'])].iloc[0].to_dict();assert p['candidate_name']==name
 if name in ['戴光宗','曾亮哲']:
  # These captures contain a document title and app shell, not an actual biography.
  assert not any(t in text for t in ['萬能工商','吳鳳技術學院'])
  audit.append(dict(candidate_key=p['candidate_key'],candidate_name=name,source_url=r['url'],source_file=r['path'],source_sha256=r['sha256'],result='page_title_only_no_readable_profile_fields',note='HTTP200與title姓名命中不當作人物履歷；不抄第三方摘要。'));continue
 ed=ex='';year='captured_2026_source_term_2022';tier='government_published_profile_not_degree_certification';locator=''
 if 'moi.gov.tw' in r['url']:
  def moi(caption):
   n=s.find('div',class_='caption',string=caption);assert n is not None
   return n.parent.select_one('.p').get_text(' | ',strip=True)
  ed=moi('學歷');ex=moi('簡歷');locator='div.essay: caption 學歷／簡歷, div.p'
 elif name=='戴德滿':
  ed='愛華國小，金城國中，第一士校，陸軍官校憲兵科';ex='桃園112憲兵隊少校調查官，東森警備保全股份有限公司負責人，桃園縣金門同鄉會理事長';year='2012';tier='government_hosted_newspaper_profile_not_government_personnel_certification';locator='人物小檔案：學歷／經歷；2012-05-05報導'
  for value in [ed,ex]:assert norm(value) in norm(text)
 elif name=='謝昌年':
  ed='育達科技大學休閒事業管理學系';ex='銅鑼鄉第17.18.19屆代表,銅鑼鄉第18屆主席';locator='鄉長簡介 .main 學歷／經歷'
  for value in [ed,ex]:assert norm(value) in norm(text)
 elif name=='詹錦章':
  ed='高工補校畢業';ex='卓蘭鎮農會理事長 | 卓蘭鎮農會常務監事 | 新榮里第19、20屆里長';locator='鎮長簡介 .main 姓名後至個人感言前'
  for value in [ed,*ex.split(' | ')]:assert norm(value) in norm(text)
 elif name=='杜文卿':
  def ly(title):
   node=s.find('h4',string=title);assert node
   return node.find_next_sibling('ul').get_text(' | ',strip=True)
  ed=ly('學歷');ex=ly('經歷');year='2002-2005_term5';locator='h4.title 學歷／經歷後ul；第5屆歷屆委員'
 assert ed and ex and norm(name) in norm(text)
 ed=ed.replace('\r','').replace('\n','\\n');ex=ex.replace('\r','').replace('\n','\\n')
 profiles.append(dict(candidate_key=p['candidate_key'],candidate_name=name,county_city=p['county_city'],electoral_district=p['electoral_district'],profile_url=r['url'],education_raw=ed,experience_raw=ex,source_year=year,evidence_tier=tier,identity_status='source_name_county_context_reviewed_independent_linkage_pending',source_file=r['path'],source_sha256=r['sha256']))
 audit.append(dict(candidate_key=p['candidate_key'],candidate_name=name,source_url=r['url'],source_file=r['path'],source_sha256=r['sha256'],result='explicit_education_and_experience_fields_read',note='保留原文字；未從軍校、姓名、照片或親屬角色推定性別。'))
 for field,value in [('profile',r['url']),('education',ed),('experience',ex)]:evidence_row(p,field,value,r['url'],locator,year,tier)
 mask=council.candidate_key.eq(p['candidate_key']);assert mask.sum()==1
 assert council.loc[mask,'education_raw'].eq('').all() and council.loc[mask,'experience_raw'].eq('').all()
 council.loc[mask,['education_raw','experience_raw','background_source_url','additional_profile_url','additional_profile_status']]=[ed,ex,r['url'],r['url'],'官方刊載履歷／政府報社人物稿；來源年代與身分另核對，非學位認證']
 update(p['candidate_key'],'profile',r['url'],r['url'],'已取得可讀人物頁；身分與來源年代待獨立複核','明確學經歷欄位已擷取；公所現職與本次參選職務分開。')
 update(p['candidate_key'],'education',ed,r['url'],'已有官方刊載學歷原文；學位未另認證','保留原文；杜文卿研究所研究不當作碩士畢業。')
# Du: explicit historical sex and prior elective offices. No DOB inferred.
p=council[council.candidate_name.eq('杜文卿')].iloc[0].to_dict();url=next(x['profile_url'] for x in profiles if x['candidate_name']=='杜文卿');text=soup_for(lookup[url]).get_text(' ',strip=True)
party=next(r for r in new_manifest if r['label']=='杜文卿2026黨公告');party_text=soup_for(party).get_text(' ',strip=True)
assert '性別：男' in text and '苗栗縣議員' in text and '議員候選人杜文卿' in party_text
identity='same_name_county_party_2026_nomination_and_historical_office_context_supported'
for field,value in [('sex','男'),('first_time_this_office','0'),('first_time_any_office','0')]:
 evidence_row(p,field,value,url,'第5屆委員性別欄／經歷苗栗縣議員及國大代表；2026黨公告另存', '2002-2005_term5','official_historical_profile_explicit_field',identity)
 update(p['candidate_key'],field,value,url+' | '+party['url'],'官方歷史人物頁與2026參選情境支持','歷史履歷明載；0=非首次，非現任推定；生日仍未知。')
for col in ['sex_additional_source_value','sex_additional_source_url','sex_additional_source_year']:council[col]=''
mask=council.candidate_key.eq(p['candidate_key']);council.loc[mask,['sex_additional_source_value','sex_additional_source_url','sex_additional_source_year','first_time_this_office_supported','first_time_any_office_supported']]=['男',url,'2002-2005','0','0']
# Chen: inspect actual 2015 bulletin cell and cross-check school/career with current official LY profile.
p=mayor[mayor.candidate_name.eq('陳素月')].iloc[0].to_dict();r=next(r for r in new_manifest if r['label']=='陳素月2015公報');ly=next(r for r in new_manifest if r['label']=='陳素月立院')
doc=pdfium.PdfDocument(ROOT/r['path']);page=doc[0];textpage=page.get_textpage();pdftext=textpage.get_text_range();textpage.close();page.close();doc.close();pdfnorm=norm(pdftext);lytext=norm(soup_for(ly).get_text(' ',strip=True))
assert '陳素月女55年1月18日' in pdfnorm
for phrase in ['文化大學史學研究所碩士','彰化縣議員']:
 assert norm(phrase) in pdfnorm and norm(phrase) in lytext
assert '彰化縣第4選舉區' in lytext
assert p['birth_date_verified']==''
mask=mayor.candidate_key.eq(p['candidate_key']);mayor.loc[mask,['birth_date_verified','age_2026_minus_birth_year','birth_date_source_url','birth_date_evidence_status']]=['1966-01-18','60',r['url'],'2015_official_bulletin_cell_visually_checked_current_LY_school_career_corroborated']
evidence_row(p,'birth_date','1966-01-18',r['url'],'2015補選公報第1頁，號次1陳素月整列；已查看渲染圖，55年1月18日；LY履歷比對','2015','official_bulletin_historical_candidate_self_declaration','name_constituency_education_and_career_corroborated_current_LY_profile')
update(p['candidate_key'],'birth_date','1966-01-18',r['url']+' | '+ly['url'],'官方公報生日欄已視讀並以立院履歷交叉核對','2015歷史申報，1966-01-18；非戶籍認證。未誤連宜蘭同名者。')
# Verify exact authorized scope before writing.
changed_keys=set(x['candidate_key'] for x in profiles)|{p['candidate_key']}
for field in ['incumbency','website','Facebook','Instagram','Threads','posts_Facebook','posts_Instagram','posts_Threads','followers','qualification','administrative_experience','political_family']:
 a=old_f[old_f.field.eq(field)].reset_index(drop=True);b=fields[fields.field.eq(field)].reset_index(drop=True);pd.testing.assert_frame_equal(a,b)
pd.testing.assert_frame_equal(old_c[['candidate_key','birth_date_supported_history','sex_supported_history']],council[['candidate_key','birth_date_supported_history','sex_supported_history']])
assert len(master)==1583 and master.candidate_key.is_unique and len(fields)==31660 and not fields.duplicated(['candidate_key','field']).any()
assert len(profiles)==7 and len(audit)==9 and len(mayor)==81 and mayor.birth_date_verified.ne('').sum()==57
assert council.birth_date_supported_history.ne('').sum()==690 and council.sex_supported_history.ne('').sum()==690
assert council.birth_date_self_declared.ne('').sum()==1
no_profile=int((council.moi_profile_url.eq('')&council.campaign_profile_url.eq('')&council.additional_profile_url.eq('')).sum());assert no_profile==756
out={'data/processed/government_profiles_round4_20261019.csv':pd.DataFrame(profiles),'data/processed/government_evidence_round4_20261019.csv':pd.DataFrame(evidence),'data/processed/government_access_review_round4_20261019.csv':pd.DataFrame(audit),'data/processed/candidates_2026_enriched_round4_20261019.csv':mayor,'data/processed/snapshots/pre_bulletin_round5_20261019/candidate_master_current.csv':master,'data/processed/snapshots/pre_bulletin_round5_20261019/candidate_fields_current.csv':fields,'data/processed/snapshots/pre_bulletin_round5_20261019/council_background_20261019.csv':council}
verification={}
for path,frame in out.items():
 target=(ROOT/path).resolve();assert target.is_relative_to((ROOT/'data/processed').resolve()) and path not in PINS
 frame.to_csv(target,index=False,encoding='utf-8-sig');saved=read(path);assert len(saved)==len(frame) and list(saved.columns)==list(frame.columns)
 assert len(target.read_text(encoding='utf-8-sig').splitlines())==len(saved)+1
 if 'field' in saved:
  keys=['candidate_key','field']+(['source_url'] if 'source_url' in saved else []);assert not saved.duplicated(keys).any()
 else:assert saved.candidate_key.is_unique
 verification[path]={'rows':len(saved),'sha256':sha(target)}
summary={'new_background_profile_people':7,'new_education_people':7,'new_experience_people':7,'page_shells_not_accepted':2,'mayor_verified_full_birth_dates':57,'mayor_missing_full_birth_dates':24,'additional_council_historical_sex_people':1,'no_profile_after_round4':756,'outputs':verification}
(ROOT/'data/code/government_profiles_round4_20261019_verification.json').write_text(json.dumps(summary,ensure_ascii=False,indent=2))
print(json.dumps(summary,ensure_ascii=False,indent=2))


{
  "new_background_profile_people": 7,
  "new_education_people": 7,
  "new_experience_people": 7,
  "page_shells_not_accepted": 2,
  "mayor_verified_full_birth_dates": 57,
  "mayor_missing_full_birth_dates": 24,
  "additional_council_historical_sex_people": 1,
  "no_profile_after_round4": 756,
  "outputs": {
    "data/processed/government_profiles_round4_20261019.csv": {
      "rows": 7,
      "sha256": "c66f99647bba2c81b336155c9d3205fab3eab57fd6bd302d31ff669b52ada04e"
    },
    "data/processed/government_evidence_round4_20261019.csv": {
      "rows": 25,
      "sha256": "8379a828fa123f02bf5331055d9e571a0789fa9ead8a4b71a597086e882bb1ad"
    },
    "data/processed/government_access_review_round4_20261019.csv": {
      "rows": 9,
      "sha256": "fdc97df6d2b75bb1d28b10e12dff6ece61b9c51563752ff74bc541f14883b96a"
    },
    "data/processed/candidates_2026_enriched_round4_20261019.csv": {
      "rows": 81,
      "sha256": "37fa15d5298ba12857f55fdd87d3d3903a1dc68d7eb975995b6d228f92be4194"


## Saved output review

In [2]:
display(read('data/processed/government_profiles_round4_20261019.csv'));display(read('data/processed/government_access_review_round4_20261019.csv'));display(read('data/processed/government_evidence_round4_20261019.csv'))

,candidate_key,candidate_name,county_city,electoral_district,profile_url,education_raw,experience_raw,source_year,evidence_tier,identity_status,source_file,source_sha256
0,2026|縣市議員|南投縣第1選舉區|廖詒綉,廖詒綉,南投縣,南投縣第1選舉區,https://www.moi.gov.tw/LocalOfficial_Content.a...,亞洲大學經營管理學系碩士,1.南投市民代表會第9~12屆市民代表\n2.中國國民黨第13、14屆全國青工總會總會長\n...,captured_2026_source_term_2022,government_published_profile_not_degree_certif...,source_name_county_context_reviewed_independen...,data/raw/background_expansion_20261019/4a24327...,cae49d6cae15158c44541bf558932315d4dd3d445867d5...
1,2026|縣市議員|金門縣第1選舉區|戴德滿,戴德滿,金門縣,金門縣第1選舉區,https://www.kmdn.gov.tw/1117/1271/4189/209125/,愛華國小，金城國中，第一士校，陸軍官校憲兵科,桃園112憲兵隊少校調查官，東森警備保全股份有限公司負責人，桃園縣金門同鄉會理事長,2012,government_hosted_newspaper_profile_not_govern...,source_name_county_context_reviewed_independen...,data/raw/background_expansion_20261019/97facb5...,4296d775f9340dfd53666f707730d164da8d6f4af74243...
2,2026|縣市議員|嘉義縣第2選舉區|何秀英,何秀英,嘉義縣,嘉義縣第2選舉區,https://www.moi.gov.tw/LocalOfficial_Content.a...,新港國中畢業,"新港鄉農會第14,15屆理事\n新港鄉民代表\n第三屆嘉南農田水利會會務委員",captured_2026_source_term_2022,government_published_profile_not_degree_certif...,source_name_county_context_reviewed_independen...,data/raw/background_expansion_20261019/5379b34...,1a56d44af9c7f3187dec4ac02b3ab7e1d69b70895e2bac...
3,2026|縣市議員|嘉義縣第3選舉區|陳靖雅,陳靖雅,嘉義縣,嘉義縣第3選舉區,https://www.moi.gov.tw/LocalOfficial_Content.a...,南華大學旅遊事業管理學系,一、專任導遊。\n二、陳信文議員服務處主任。,captured_2026_source_term_2022,government_published_profile_not_degree_certif...,source_name_county_context_reviewed_independen...,data/raw/background_expansion_20261019/b59bfa2...,105494a4ed81a80e77a3a4fb56d9ad80f1651bd5ebab2b...
4,2026|縣市議員|苗栗縣第2選舉區|謝昌年,謝昌年,苗栗縣,苗栗縣第2選舉區,https://www.tongluo.gov.tw/cp.aspx?n=3341,育達科技大學休閒事業管理學系,"銅鑼鄉第17.18.19屆代表,銅鑼鄉第18屆主席",captured_2026_source_term_2022,government_published_profile_not_degree_certif...,source_name_county_context_reviewed_independen...,data/raw/background_expansion_20261019/2a8a84d...,5e5b30590457d9f2605b279e3b56e9b8449c20b6d14acd...
5,2026|縣市議員|苗栗縣第3選舉區|杜文卿,杜文卿,苗栗縣,苗栗縣第3選舉區,https://www.ly.gov.tw/Pages/List.aspx?nodeid=1126,美國加州大學長堤分校國際社工研究所研究,民進黨苗栗縣黨部主委、 | 民進黨中央執行委員、 | 民進黨中央評議委員、 | 國代主席團主...,2002-2005_term5,government_published_profile_not_degree_certif...,source_name_county_context_reviewed_independen...,data/raw/background_expansion_20261019/f040dc2...,8df4a341482cdb1791793fb34ccbffd52199e788b09924...
6,2026|縣市議員|苗栗縣第6選舉區|詹錦章,詹錦章,苗栗縣,苗栗縣第6選舉區,https://www.juolan.gov.tw/cp.aspx?n=3137,高工補校畢業,卓蘭鎮農會理事長 | 卓蘭鎮農會常務監事 | 新榮里第19、20屆里長,captured_2026_source_term_2022,government_published_profile_not_degree_certif...,source_name_county_context_reviewed_independen...,data/raw/background_expansion_20261019/db2ca39...,f8afa62fffbac349bf1d4054a6e4d9efb433c96e6e634e...


,candidate_key,candidate_name,source_url,source_file,source_sha256,result,note
0,2026|縣市議員|南投縣第1選舉區|廖詒綉,廖詒綉,https://www.moi.gov.tw/LocalOfficial_Content.a...,data/raw/background_expansion_20261019/4a24327...,cae49d6cae15158c44541bf558932315d4dd3d445867d5...,explicit_education_and_experience_fields_read,保留原文字；未從軍校、姓名、照片或親屬角色推定性別。
1,2026|縣市議員|金門縣第1選舉區|戴德滿,戴德滿,https://www.kmdn.gov.tw/1117/1271/4189/209125/,data/raw/background_expansion_20261019/97facb5...,4296d775f9340dfd53666f707730d164da8d6f4af74243...,explicit_education_and_experience_fields_read,保留原文字；未從軍校、姓名、照片或親屬角色推定性別。
2,2026|縣市議員|嘉義縣第2選舉區|何秀英,何秀英,https://www.moi.gov.tw/LocalOfficial_Content.a...,data/raw/background_expansion_20261019/5379b34...,1a56d44af9c7f3187dec4ac02b3ab7e1d69b70895e2bac...,explicit_education_and_experience_fields_read,保留原文字；未從軍校、姓名、照片或親屬角色推定性別。
3,2026|縣市議員|嘉義縣第3選舉區|陳靖雅,陳靖雅,https://www.moi.gov.tw/LocalOfficial_Content.a...,data/raw/background_expansion_20261019/b59bfa2...,105494a4ed81a80e77a3a4fb56d9ad80f1651bd5ebab2b...,explicit_education_and_experience_fields_read,保留原文字；未從軍校、姓名、照片或親屬角色推定性別。
4,2026|縣市議員|苗栗縣第2選舉區|謝昌年,謝昌年,https://www.tongluo.gov.tw/cp.aspx?n=3341,data/raw/background_expansion_20261019/2a8a84d...,5e5b30590457d9f2605b279e3b56e9b8449c20b6d14acd...,explicit_education_and_experience_fields_read,保留原文字；未從軍校、姓名、照片或親屬角色推定性別。
5,2026|縣市議員|苗栗縣第3選舉區|杜文卿,杜文卿,https://www.ly.gov.tw/Pages/List.aspx?nodeid=1126,data/raw/background_expansion_20261019/f040dc2...,8df4a341482cdb1791793fb34ccbffd52199e788b09924...,explicit_education_and_experience_fields_read,保留原文字；未從軍校、姓名、照片或親屬角色推定性別。
6,2026|縣市議員|苗栗縣第6選舉區|詹錦章,詹錦章,https://www.juolan.gov.tw/cp.aspx?n=3137,data/raw/background_expansion_20261019/db2ca39...,f8afa62fffbac349bf1d4054a6e4d9efb433c96e6e634e...,explicit_education_and_experience_fields_read,保留原文字；未從軍校、姓名、照片或親屬角色推定性別。
7,2026|縣市議員|嘉義縣第1選舉區|戴光宗,戴光宗,https://www.cyscc.gov.tw/Parliamentary_Content...,data/raw/background_expansion_20261019/8b852a1...,e0f28b71bde3fe6e777d0ec983ecbe735c531341086ef8...,page_title_only_no_readable_profile_fields,HTTP200與title姓名命中不當作人物履歷；不抄第三方摘要。
8,2026|縣市議員|嘉義縣第6選舉區|曾亮哲,曾亮哲,https://www.cyscc.gov.tw/Parliamentary_Content...,data/raw/background_expansion_20261019/a5f3414...,fd478762711a58d7f96988e0e3a6ad983401ed373d700b...,page_title_only_no_readable_profile_fields,HTTP200與title姓名命中不當作人物履歷；不抄第三方摘要。


,candidate_key,candidate_name,county_city,field,value_raw,source_url,source_file,source_sha256,source_locator,source_year,evidence_tier,identity_status,checked_date
0,2026|縣市議員|南投縣第1選舉區|廖詒綉,廖詒綉,南投縣,profile,https://www.moi.gov.tw/LocalOfficial_Content.a...,https://www.moi.gov.tw/LocalOfficial_Content.a...,data/raw/background_expansion_20261019/4a24327...,cae49d6cae15158c44541bf558932315d4dd3d445867d5...,"div.essay: caption 學歷／簡歷, div.p",captured_2026_source_term_2022,government_published_profile_not_degree_certif...,source_name_county_context_reviewed_independen...,2026-10-19
1,2026|縣市議員|南投縣第1選舉區|廖詒綉,廖詒綉,南投縣,education,亞洲大學經營管理學系碩士,https://www.moi.gov.tw/LocalOfficial_Content.a...,data/raw/background_expansion_20261019/4a24327...,cae49d6cae15158c44541bf558932315d4dd3d445867d5...,"div.essay: caption 學歷／簡歷, div.p",captured_2026_source_term_2022,government_published_profile_not_degree_certif...,source_name_county_context_reviewed_independen...,2026-10-19
2,2026|縣市議員|南投縣第1選舉區|廖詒綉,廖詒綉,南投縣,experience,1.南投市民代表會第9~12屆市民代表\n2.中國國民黨第13、14屆全國青工總會總會長\n...,https://www.moi.gov.tw/LocalOfficial_Content.a...,data/raw/background_expansion_20261019/4a24327...,cae49d6cae15158c44541bf558932315d4dd3d445867d5...,"div.essay: caption 學歷／簡歷, div.p",captured_2026_source_term_2022,government_published_profile_not_degree_certif...,source_name_county_context_reviewed_independen...,2026-10-19
3,2026|縣市議員|金門縣第1選舉區|戴德滿,戴德滿,金門縣,profile,https://www.kmdn.gov.tw/1117/1271/4189/209125/,https://www.kmdn.gov.tw/1117/1271/4189/209125/,data/raw/background_expansion_20261019/97facb5...,4296d775f9340dfd53666f707730d164da8d6f4af74243...,人物小檔案：學歷／經歷；2012-05-05報導,2012,government_hosted_newspaper_profile_not_govern...,source_name_county_context_reviewed_independen...,2026-10-19
4,2026|縣市議員|金門縣第1選舉區|戴德滿,戴德滿,金門縣,education,愛華國小，金城國中，第一士校，陸軍官校憲兵科,https://www.kmdn.gov.tw/1117/1271/4189/209125/,data/raw/background_expansion_20261019/97facb5...,4296d775f9340dfd53666f707730d164da8d6f4af74243...,人物小檔案：學歷／經歷；2012-05-05報導,2012,government_hosted_newspaper_profile_not_govern...,source_name_county_context_reviewed_independen...,2026-10-19
5,2026|縣市議員|金門縣第1選舉區|戴德滿,戴德滿,金門縣,experience,桃園112憲兵隊少校調查官，東森警備保全股份有限公司負責人，桃園縣金門同鄉會理事長,https://www.kmdn.gov.tw/1117/1271/4189/209125/,data/raw/background_expansion_20261019/97facb5...,4296d775f9340dfd53666f707730d164da8d6f4af74243...,人物小檔案：學歷／經歷；2012-05-05報導,2012,government_hosted_newspaper_profile_not_govern...,source_name_county_context_reviewed_independen...,2026-10-19
6,2026|縣市議員|嘉義縣第2選舉區|何秀英,何秀英,嘉義縣,profile,https://www.moi.gov.tw/LocalOfficial_Content.a...,https://www.moi.gov.tw/LocalOfficial_Content.a...,data/raw/background_expansion_20261019/5379b34...,1a56d44af9c7f3187dec4ac02b3ab7e1d69b70895e2bac...,"div.essay: caption 學歷／簡歷, div.p",captured_2026_source_term_2022,government_published_profile_not_degree_certif...,source_name_county_context_reviewed_independen...,2026-10-19
7,2026|縣市議員|嘉義縣第2選舉區|何秀英,何秀英,嘉義縣,education,新港國中畢業,https://www.moi.gov.tw/LocalOfficial_Content.a...,data/raw/background_expansion_20261019/5379b34...,1a56d44af9c7f3187dec4ac02b3ab7e1d69b70895e2bac...,"div.essay: caption 學歷／簡歷, div.p",captured_2026_source_term_2022,government_published_profile_not_degree_certif...,source_name_county_context_reviewed_independen...,2026-10-19
8,2026|縣市議員|嘉義縣第2選舉區|何秀英,何秀英,嘉義縣,experience,"新港鄉農會第14,15屆理事\n新港鄉民代表\n第三屆嘉南農田水利會會務委員",https://www.moi.gov.tw/LocalOfficial_Content.a...,data/raw/background_expansion_20261019/5379b34...,1a56d44af9c7f3187dec4ac02b3ab7e1d69b70895e2bac...,"div.essay: caption 學歷／簡歷, div.p",captured_2026_source_term_2022,government_published_profile_not_degree_certif...,source_name_county_context_reviewed_independen...,2026-10-19
9,2026|縣市議員|嘉義縣第3選舉區|陳靖雅,陳靖雅,嘉義縣,profile,https://www.moi.gov.tw/LocalOfficial_Content.a...,https://www.moi.gov.tw/LocalOfficial_Content.a...,data/raw/background_expansion_20261019/b59bfa2...,105494a4ed81a80e77a3a4fb56d9ad80f1651bd5ebab2b...,"div.essay: caption 學歷／簡歷, div.p",captured_2026_source_term_2022,go

2026-10-20 維護：重複檔案改用單一保存路徑，固定指紋同步更新；議員登記CSV排版換行改為空格，政黨標籤不變。本次已獨立從raw表格重建並比對全部1,502筆及16欄；未重跑本Notebook或整條研究流程，既有執行輸出屬先前執行紀錄。